# **Median Filter L4 + Pure DWT L4 (เฉพาะส่วนที่ขาด)**

ใช้การตั้งค่าเดียวกับ `Seminae_source_code (1).ipynb` ทุกอย่าง:
- Dataset: `awsaf49/vinbigdata-512-image-dataset`, Infiltration vs No finding
- Split: Train 100:100, Test 50:50, Val 20:20 (`random_state=42` ลำดับเดียวกับต้นฉบับ)
- ResNet-50 ImageNet, Adam lr=1e-4, batch 32, 7 epochs
- **Median L4** = `cv2.medianBlur` kernel 9x9
- **Pure DWT L4** = db1 soft threshold ratio 0.35 (resize 224 ก่อน DWT เหมือนต้นฉบับ)

Runtime > Change runtime type > **GPU** แล้วกด Run all

In [ ]:
!pip -q install kagglehub PyWavelets

In [ ]:
import os
import glob
import random
import cv2
import pywt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import kagglehub

# ต้นฉบับไม่ได้ fix seed ของ torch -> ตั้ง SEED = None เพื่อให้เหมือนต้นฉบับ
# ตั้งค่าเป็นตัวเลข (เช่น 42) ถ้าต้องการผลที่รันซ้ำได้
SEED = 42
if SEED is not None:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"กำลังประมวลผลบน: {device}")

In [ ]:
# -------------------------------------------------------------
# 1. โหลด Dataset และแบ่งชุดข้อมูล (เหมือนต้นฉบับทุกขั้นตอน)
# -------------------------------------------------------------
base_path = kagglehub.dataset_download('awsaf49/vinbigdata-512-image-dataset')
print("Path to images:", base_path)

train_dir = os.path.join(base_path, 'vinbigdata', 'train')
if not os.path.exists(train_dir):
    train_dir = os.path.join(base_path, 'train')

csv_files = glob.glob(os.path.join(base_path, '**/*train.csv'), recursive=True)
csv_path = csv_files[0] if csv_files else glob.glob(os.path.join(base_path, '**/*.csv'), recursive=True)[0]

print(f"ใช้ไฟล์ CSV จาก: {csv_path}")
print(f"โฟลเดอร์รูปภาพ Train: {train_dir}")

df = pd.read_csv(csv_path)

if 'class_name' in df.columns:
    df_binary = df[df['class_name'].isin(['Infiltration', 'No finding'])].copy()
    df_binary['target'] = (df_binary['class_name'] == 'Infiltration').astype(int)
else:
    df_binary = df.copy()
    df_binary['target'] = (df_binary['class_id'] != 14).astype(int)

df_unique = df_binary.drop_duplicates(subset=['image_id']).copy()
df_unique['img_path'] = df_unique['image_id'].apply(lambda x: os.path.join(train_dir, f"{x}.png"))
df_unique = df_unique[df_unique['img_path'].apply(os.path.exists)].reset_index(drop=True)

df_class_0 = df_unique[df_unique['target'] == 0]
df_class_1 = df_unique[df_unique['target'] == 1]
print(f"จำนวนรูปที่พบทั้งหมด -> Normal: {len(df_class_0)} ภาพ | Infiltration: {len(df_class_1)} ภาพ")

train_df_0 = df_class_0.sample(n=100, random_state=42)
train_df_1 = df_class_1.sample(n=100, random_state=42)
train_df = pd.concat([train_df_0, train_df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

remaining_0 = df_class_0.drop(train_df_0.index)
remaining_1 = df_class_1.drop(train_df_1.index)

test_df_0 = remaining_0.sample(n=50, random_state=42)
test_df_1 = remaining_1.sample(n=50, random_state=42)
test_df = pd.concat([test_df_0, test_df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

remaining_0 = remaining_0.drop(test_df_0.index)
remaining_1 = remaining_1.drop(test_df_1.index)
val_df_0 = remaining_0.sample(n=20, random_state=42)
val_df_1 = remaining_1.sample(n=20, random_state=42)
val_df = pd.concat([val_df_0, val_df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"- Train: {len(train_df)} ภาพ (Normal: {(train_df['target']==0).sum()}, Infiltration: {(train_df['target']==1).sum()})")
print(f"- Val  : {len(val_df)} ภาพ (Normal: {(val_df['target']==0).sum()}, Infiltration: {(val_df['target']==1).sum()})")
print(f"- Test : {len(test_df)} ภาพ (Normal: {(test_df['target']==0).sum()}, Infiltration: {(test_df['target']==1).sum()})")
# ต้นฉบับ: Normal 10606 | Infiltration 613 -> ถ้าตัวเลขตรง แปลว่า split เหมือนเดิม

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# -------------------------------------------------------------
# 2. ฟังก์ชันฝึกสอน + ประเมินผล (ลูปเดียวกับต้นฉบับ: ResNet-50, 7 Epochs)
# -------------------------------------------------------------
def train_and_evaluate(name, train_ds, test_ds, save_name):
    print(f"\n>>> [กำลังเริ่มการทดลอง: {name}] <<<")

    lvl_train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    lvl_test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

    lvl_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    lvl_model.fc = nn.Linear(lvl_model.fc.in_features, 2)
    lvl_model = lvl_model.to(device)

    lvl_criterion = nn.CrossEntropyLoss()
    lvl_optimizer = torch.optim.Adam(lvl_model.parameters(), lr=1e-4)

    lvl_model.train()
    for epoch in range(7):
        running_loss, correct, total = 0.0, 0, 0
        for imgs, lbls in lvl_train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            lvl_optimizer.zero_grad()
            outputs = lvl_model(imgs)
            loss = lvl_criterion(outputs, lbls)
            loss.backward()
            lvl_optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == lbls.data).item()
            total += lbls.size(0)

        if (epoch + 1) in [1, 4, 7]:
            print(f"Epoch [{epoch+1:02d}/07] Loss: {running_loss/total:.4f} | Train Acc: {correct/total:.4f}")

    torch.save(lvl_model.state_dict(), save_name)
    print(f"บันทึกโมเดล {save_name} เรียบร้อย")

    lvl_model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in lvl_test_loader:
            imgs = imgs.to(device)
            outputs = lvl_model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(lbls.numpy())

    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    row = {
        'สภาวะการทดลอง': name,
        'Accuracy': accuracy_score(all_targets, all_preds),
        'Precision': precision_score(all_targets, all_preds, zero_division=0),
        'Recall': recall_score(all_targets, all_preds, zero_division=0),
        'F1-Score': f1_score(all_targets, all_preds, zero_division=0),
        'ROC-AUC': roc_auc_score(all_targets, all_probs)
    }
    return row, confusion_matrix(all_targets, all_preds)


results = []
confusion_matrices = {}

# **Median Filter L4 (9x9)**

In [ ]:
def apply_median_filter(pil_img, level='L1'):
    img_gray = np.array(pil_img.convert('L'))
    kernel_sizes = {'L1': 3, 'L2': 5, 'L3': 7, 'L4': 9}
    ksize = kernel_sizes.get(level, 3)
    filtered = cv2.medianBlur(img_gray, ksize)
    return Image.fromarray(filtered).convert('RGB')

class DenoisedCXRDataset(Dataset):
    def __init__(self, dataframe, filter_level='L1', transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.filter_level = filter_level
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'img_path']
        image = Image.open(img_path).convert('RGB')
        image = apply_median_filter(image, self.filter_level)
        label = int(self.df.loc[idx, 'target'])

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

row, cm = train_and_evaluate(
    'Median Filter L4',
    DenoisedCXRDataset(train_df, filter_level='L4', transform=train_transform),
    DenoisedCXRDataset(test_df, filter_level='L4', transform=val_test_transform),
    'median_l4_resnet50.pth'
)
results.append(row)
confusion_matrices['median_l4'] = cm
display(pd.DataFrame([row]))

# **Pure DWT L4 (threshold ratio 0.35)**

In [ ]:
dwt_threshold_ratios = {'L1': 0.05, 'L2': 0.10, 'L3': 0.20, 'L4': 0.35}

def apply_pure_dwt(img_gray_np, thresh_ratio=0.10):
    coeffs2 = pywt.dwt2(img_gray_np.astype(np.float32), 'db1')
    LL, (LH, HL, HH) = coeffs2

    max_coeff = np.max([np.abs(LH), np.abs(HL), np.abs(HH)])
    threshold = thresh_ratio * max_coeff

    LH_t = pywt.threshold(LH, threshold, mode='soft')
    HL_t = pywt.threshold(HL, threshold, mode='soft')
    HH_t = pywt.threshold(HH, threshold, mode='soft')

    recon = pywt.idwt2((LL, (LH_t, HL_t, HH_t)), 'db1')
    return np.clip(recon, 0, 255).astype(np.uint8)

def cache_dwt_dataset(df_subset, subset_name, levels=('L4',)):
    for lvl in levels:
        os.makedirs(f"/content/dwt_cache/{subset_name}/{lvl}", exist_ok=True)

    print(f"กำลังบันทึกแคช DWT สำหรับชุด {subset_name} ({len(df_subset)} ภาพ)...")
    for _, row in df_subset.iterrows():
        img_id = row['image_id']
        img_gray = cv2.imread(row['img_path'], cv2.IMREAD_GRAYSCALE)
        img_resized = cv2.resize(img_gray, (224, 224))

        for lvl in levels:
            denoised_dwt = apply_pure_dwt(img_resized, thresh_ratio=dwt_threshold_ratios[lvl])
            cv2.imwrite(f"/content/dwt_cache/{subset_name}/{lvl}/{img_id}.png", denoised_dwt)

cache_dwt_dataset(train_df, "train")
cache_dwt_dataset(test_df, "test")
print("แคชชุดข้อมูล DWT L4 เสร็จสมบูรณ์!")

class CachedDWTCXRDataset(Dataset):
    def __init__(self, dataframe, subset_name="train", level='L1', transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.subset_name = subset_name
        self.level = level
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'image_id']
        cached_path = f"/content/dwt_cache/{self.subset_name}/{self.level}/{img_id}.png"
        image = Image.open(cached_path).convert('RGB')
        label = int(self.df.loc[idx, 'target'])

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

row, cm = train_and_evaluate(
    'DWT L4',
    CachedDWTCXRDataset(train_df, "train", level='L4', transform=train_transform),
    CachedDWTCXRDataset(test_df, "test", level='L4', transform=val_test_transform),
    'dwt_l4_resnet50.pth'
)
results.append(row)
confusion_matrices['dwt_l4'] = cm
display(pd.DataFrame([row]))

# **สรุปผล L4**

In [ ]:
df_results = pd.DataFrame(results)
print("=" * 70)
print(" สรุปผลการประเมิน L4 (Median Filter + Pure DWT)")
print("=" * 70)
display(df_results.style.format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}',
    'Recall': '{:.4f}', 'F1-Score': '{:.4f}', 'ROC-AUC': '{:.4f}'
}))
df_results.to_csv('results_L4_median_dwt.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, len(confusion_matrices), figsize=(11, 5))
class_labels = ['Normal', 'Infiltration']

for idx, (name, cm) in enumerate(confusion_matrices.items()):
    ax = axes[idx]
    ax.imshow(cm, cmap=plt.cm.Blues, interpolation='nearest')
    ax.set_title(name, fontsize=14, pad=12)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(class_labels, fontsize=11)
    ax.set_yticklabels(class_labels, fontsize=11)
    ax.set_xlabel('Predicted (count / % of actual class)', fontsize=11)
    if idx == 0:
        ax.set_ylabel('Actual', fontsize=11)

    for i in range(2):
        row_sum = cm[i].sum()
        for j in range(2):
            count = cm[i, j]
            pct = (count / row_sum * 100) if row_sum > 0 else 0
            text_color = "white" if count > (cm.max() / 2) else "black"
            ax.text(j, i, f"{count}\n({pct:.2f}%)", ha="center", va="center",
                    color=text_color, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices_L4_median_dwt.png', dpi=300)
plt.show()
print("บันทึก results_L4_median_dwt.csv และ confusion_matrices_L4_median_dwt.png แล้ว")

In [ ]:
# (ไม่บังคับ) ดาวน์โหลดผลลัพธ์และ checkpoint
from google.colab import files
for f in ['results_L4_median_dwt.csv', 'confusion_matrices_L4_median_dwt.png',
          'median_l4_resnet50.pth', 'dwt_l4_resnet50.pth']:
    if os.path.exists(f):
        files.download(f)